# 4.3 Function Generators

**Prerequisites:** 4.1 Functions User-defined, 3.3 Comprehension  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What a generator is, and how `yield` suspends and resumes a function
- Generator functions vs generator expressions
- Memory and laziness — measured, not asserted
- **`yield from`** for delegation and recursive traversal
- **The full protocol**: `send()`, `close()`, `throw()`, and `return`
- Building lazy **pipelines** over large or infinite data
- How generators relate to `async`/`await`

---

### Why Generator:
- There is a lot of overhead in building an iterator in Python.
    - We have to implement a class with `__iter__()` and `__next__()` method, 
    - keep track of internal states, 
    - raise StopIteration when there was no values to be returned 
    - etc.
- This is both lengthy and counter intuitive. Generator comes into rescue in such situations.

## Generator
- Generator are the simple way of creating iterators.
    - All the overhead we mentioned above are automatically handled by generators in Python.
-  A generator is a function that returns an object (iterator) which we can iterate over (one value at a time).
<img src='./Image/4.3 Image a.png' width=80% height=60%/>

### How to create generator:
- It is as easy as defining a normal function with yield statement instead of a return statement.
- A generator has parameter, which we can called and it generates a sequence of numbers. 
    - But unlike functions, which return a whole array, a generator yields one value at a time which requires less memory.
- Inside a function if we use atleast one 'yield' keyword then that function becomes generator function.
- In case of generator when it encounters a yield keyword the state of the function is frozen and all the variables are stored in memory until the generator is called again.
    - Each time next() is called on the generator function, the generator resumes where it left off (it remembers all the data values and which statement was last executed).
<br/><br/>
-  **Summary**, Generators in Python:
    - Defined with the def keyword.
    - Use the yield keyword.
    - May contain several yield keywords.
    - Returns an iterator.
<br/><br/>
- **Syntax:**

```python
def gereratorName(argument):
   #statements
       yield argument
   #statements

#calling the generator
variableName = gereratorName(10)
print(variableName)
```
 
### Return Vs Yield statement:
- Both yield and return will return some value from a function.
- The difference is that, 
    - while a 'return' statement terminates a function entirely, 
    - yield statement pauses the function saving all its states and later continues from there on successive calls.
    

### Generator function Vs a Normal function:
- Here is how a generator function differs from a normal function:
    - Generator function contains one or more yield statement.
    - When called, it returns an object (iterator) but does not start execution immediately.
    - Methods like `__iter__()` and `__next__()` are implemented automatically. So we can iterate through the items using next().
    - Once the function yields, the function is paused and the control is transferred to the caller.
    - Local variables and their states are remembered between successive calls.
    - Finally, when the function terminates, StopIteration is raised automatically on further calls.

In [ ]:
#A generator is a function that returns an object (iterator) which we can iterate over
#generator function named seq_gen() with several yield statements.
def seq_gen():
    yield 'python'
    yield 'Java'
    yield 'C++'
    
g = seq_gen() # seq_gen() will return iterator object, saving in g
print(next(g))
print(next(g))
print(next(g))
# print(next(g))

In [ ]:
#Once the function yields, the function is paused and the control is transferred to the caller.
#Local variables and their states are remembered between successive calls.
def seq_gen():
    n=1
    print('state:', n)
    yield n
    n+=1
    print('state:', n)
    yield n
    n+=1
    print('state:', n)
    yield n
    
g = seq_gen()
print(next(g))
print(next(g))
print(next(g))
# print(next(g))

- The value of variable n is remembered between each call.
- Unlike normal functions, the local variables are not destroyed when the function yields.
- Generator functions are implemented with a loop having a suitable terminating condition.

In [ ]:
#implementation with a loop
def seq_gen():
    for i in range(1,4):
        print('state:', i)
        yield i
    
g = seq_gen()
print(next(g))
print(next(g))
print(next(g))
# print(next(g))

### Get Python Generator’s value with implicit next() call
- We can use generators with for loops directly.
- This is because, a for loop takes an iterator and iterates over it using next() function. It automatically ends when StopIteration is raised.

In [ ]:
def generator_func():
    for i in range(1,4):
        print('state:', i)
        yield i
    
for g in generator_func():
    print(g)

### WAP to get table of a number n:

In [ ]:
def timesTable(number):
    for i in range(1, 11):
        yield i * number
        i += 1

n= int(input("Enter no.: "))
gettimes = timesTable(n)
for a in gettimes:
    print(a)

### WAP to print square of first n natural no.s:

In [ ]:
# Using for loop
n= int(input("Enter range: "))
sq_arr=[]
for i in range(n):
    sq_arr.append(i*i)
print(sq_arr)

In [ ]:
def sq_gen(n):
    num=1
    while True:
        yield num
        if num==n:
            return
        else:
            num +=1

In [ ]:
n= int(input("Enter end of range: "))
sq_arr=[]
for i in sq_gen(n):
    sq_arr.append(i*i)
print(sq_arr)

- The major difference between a list comprehension and a generator expression is that while list comprehension produces the entire list, generator expression produces one item at a time.

In [ ]:
import sys

n = 100_000

list_comp = [i * i for i in range(n)]
gen_exp = (i * i for i in range(n))

print(f"list comprehension : {sys.getsizeof(list_comp):>10,} bytes")
print(f"generator expression: {sys.getsizeof(gen_exp):>9,} bytes")
print(f"ratio: about {sys.getsizeof(list_comp) / sys.getsizeof(gen_exp):,.0f}x")

# The generator's size does NOT grow with n - it stores a recipe, not results
for size in (1_000, 1_000_000, 100_000_000):
    g = (i * i for i in range(size))
    print(f"  n={size:>12,}  generator = {sys.getsizeof(g)} bytes")

# Both give the same answer
print("\nsum via list :", sum(list_comp))
print("sum via gen  :", sum(i * i for i in range(n)))

In [ ]:
import timeit

n = 100_000

# Building the whole list costs time up front
t_list = timeit.timeit(lambda: [i * i for i in range(n)], number=20)

# Creating a generator is nearly free...
t_gen_create = timeit.timeit(lambda: (i * i for i in range(n)), number=20)

# ...but consuming it costs about the same as the list did
t_gen_consume = timeit.timeit(lambda: sum(i * i for i in range(n)), number=20)

print(f"build list        : {t_list * 1000:8.2f} ms")
print(f"create generator  : {t_gen_create * 1000:8.4f} ms   <- does no work")
print(f"consume generator : {t_gen_consume * 1000:8.2f} ms")

print("""
The generator does not make the WORK cheaper - it makes the work LAZY,
and it never holds more than one item at a time. That is the trade:
same total time, dramatically less memory, and you can stop early.
""")

# Stopping early is where laziness actually wins on time too
first_over_1000 = next(i * i for i in range(n) if i * i > 1000)
print("first square over 1000:", first_over_1000, "- computed 32 items, not 100,000")

### Using Generator, WAF to print the fibonacci series:

In [ ]:
def fibo(n):
    a,b = 0,1
    print(0, end=' ')
    print(1, end=' ')
    while True:
        c = a+b
        if c<=n-2:
            yield c
            a = b
            b = c
        else:
            break

In [ ]:
n = int(input("Enter the range: "))
for i in fibo(n):
    print(i, end=' ')

### Using Generator, WAF to find out factorial Series:

In [ ]:
def factorial(n):
    fact = []
    k = 1
    for i in range(1,n+1):
        k *= i
        fact.append(k)
    return fact

n= int(input("Enter no.: "))
f= factorial(n)
print(f) 

In [ ]:
def factorial(n):
    k = 1
    for i in range(1, n+1):
        k *= i
        yield k
        
fact= []
n = int(input("Enter the number: "))
for i in factorial(n):
    fact.append(i)
print(fact)

### Suppose we have a log file from a fast food chain. 
- The log file has a column (4th column) that keeps track of the number of pizza sold every hour and we want to sum it to find the total pizzas sold in 5 years.
- Assume everything is in string and numbers that are not available are marked as 'N/A'. A generator implementation of this could be as follows.

```python
with open('sells.log') as file:
    pizza_col = (line[3] for line in file)
    per_hour = (int(x) for x in pizza_col if x != 'N/A')
    print("Total pizzas sold = ",sum(per_hour))
```

---

### `yield from` — delegating to another generator

> **Version note:** added in **Python 3.3** (PEP 380).

When a generator wants to yield everything from *another* iterable, the obvious code is a
loop that re-yields each item. `yield from` says the same thing in one line — and does more:

| | Manual loop | `yield from` |
|---|---|---|
| Yields every item | ✅ | ✅ |
| Forwards `send()` / `throw()` to the sub-generator | ❌ | ✅ |
| Captures the sub-generator's `return` value | ❌ | ✅ |

```
yield from iterable
    |
    +-- "yield everything this produces, then carry on"
```

**Real-world use case:** recursive traversal — flattening nested lists, walking a directory
tree, or visiting a tree of nodes. The recursive call is itself a generator, so `yield from`
is exactly the right verb.

In [ ]:
# ---- Without yield from: a manual re-yield loop ----
def chain_manual(*iterables):
    for it in iterables:
        for item in it:
            yield item

# ---- With yield from: delegation ----
def chain_delegated(*iterables):
    for it in iterables:
        yield from it

print("manual   :", list(chain_manual([1, 2], (3, 4), "ab")))
print("delegated:", list(chain_delegated([1, 2], (3, 4), "ab")))


# ---- Where it really pays: recursion over nested structures ----
def flatten(nested):
    """Yield every scalar in an arbitrarily nested list."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)      # delegate to the recursive call
        else:
            yield item

data = [1, [2, 3, [4, [5, 6]], 7], 8, [[9]]]
print("\nflatten:", list(flatten(data)))

# The same without yield from - noisier, and easy to get wrong
def flatten_manual(nested):
    for item in nested:
        if isinstance(item, list):
            for sub in flatten_manual(item):
                yield sub
        else:
            yield item

assert list(flatten(data)) == list(flatten_manual(data))


# ---- yield from also forwards the generator's RETURN value ----
def inner():
    yield 1
    yield 2
    return "inner done"

def outer():
    result = yield from inner()      # captures inner's return value
    print(f"  inner returned: {result!r}")
    yield 3

print("\nouter:", list(outer()))

---

### The full generator protocol: `send`, `close`, `throw`

`next()` is only one of four ways to interact with a paused generator. The others turn a
generator from something you *pull from* into something you can also *push into* — which is
what the word **coroutine** originally meant in Python.

| Method | Effect |
|---|---|
| `next(gen)` | Resume; run to the next `yield` |
| `gen.send(value)` | Resume, and make the paused `yield` **evaluate to `value`** |
| `gen.close()` | Raise `GeneratorExit` at the `yield`, so cleanup can run |
| `gen.throw(exc)` | Raise `exc` at the `yield`, so the generator can handle it |

The key insight for `send()`: **`yield` is an expression**. `value = yield total` sends
`total` out *and* waits to receive something back.

> ⚠️ A fresh generator must be advanced to its first `yield` before you can `send()` a real
> value — call `next(gen)` (or `gen.send(None)`) once to prime it.

In [ ]:
# ---- .send(): push a value INTO a paused generator ----
def accumulator():
    """A coroutine: yields the running total, receives the next addend."""
    total = 0
    while True:
        value = yield total          # yields `total`, and receives what is sent back
        if value is None:
            continue
        total += value


acc = accumulator()
print("prime it :", next(acc))       # MUST advance to the first yield first
print("send 10  :", acc.send(10))
print("send 5   :", acc.send(5))
print("send 100 :", acc.send(100))


# ---- .close(): stop a generator, raising GeneratorExit inside it ----
def managed():
    print("  acquiring resource")
    try:
        while True:
            yield "working"
    except GeneratorExit:
        print("  cleaning up")       # runs on close() - this is why `with` works in generators

g = managed()
print("\n", next(g))
g.close()


# ---- .throw(): raise an exception at the point of the yield ----
def resilient():
    while True:
        try:
            yield "ok"
        except ValueError as exc:
            print(f"  handled inside the generator: {exc}")

r = resilient()
print("\n", next(r))
print(" ", r.throw(ValueError("bad data")))


# ---- return inside a generator ----
def counter(limit):
    n = 0
    while n < limit:
        yield n
        n += 1
    return f"finished after {n}"     # goes into StopIteration.value, NOT into the loop

gen = counter(3)
print("\nvalues:", list(gen))

gen = counter(3)
while True:
    try:
        next(gen)
    except StopIteration as stop:
        print("return value:", stop.value)
        break

---

### Generator pipelines

Because a generator both *consumes* an iterable and *produces* one, generators chain. Each
stage pulls one item from the stage before it, so a million-line file flows through with
only one line in memory at a time.

**Analogy:** an assembly line rather than a warehouse. Nothing is stockpiled between stations.

**Real-world use case:** log processing, ETL, reading large CSVs, streaming API results —
anywhere the data is bigger than RAM or arrives over time.

In [ ]:
# A pipeline: each stage is lazy, so nothing is held in memory all at once.
LOG = """2024-01-01 INFO  service started
2024-01-01 ERROR db connection refused
2024-01-02 INFO  retry scheduled
2024-01-02 ERROR db connection refused
2024-01-03 WARN  slow query 2.3s
2024-01-03 ERROR disk full""".splitlines()


def read_lines(source):
    """Stage 1: yield raw lines."""
    for line in source:
        yield line


def parse(lines):
    """Stage 2: turn each line into a dict."""
    for line in lines:
        date, level, *message = line.split()
        yield {"date": date, "level": level, "message": " ".join(message)}


def only(records, level):
    """Stage 3: filter."""
    for r in records:
        if r["level"] == level:
            yield r


# Wiring the stages together does NO work yet
pipeline = only(parse(read_lines(LOG)), "ERROR")
print("pipeline object:", type(pipeline).__name__, "- nothing has run")

print("\nnow consuming:")
for record in pipeline:
    print(f"  {record['date']}  {record['message']}")

# The same thing with generator expressions
records = ({"level": l.split()[1], "raw": l} for l in LOG)
errors = (r for r in records if r["level"] == "ERROR")
print("\nerror count:", sum(1 for _ in errors))


# ---- Infinite generators are fine, because you control consumption ----
import itertools

def naturals():
    n = 1
    while True:
        yield n
        n += 1

print("\nfirst 5 naturals :", list(itertools.islice(naturals(), 5)))
print("first 5 squares  :", [n * n for n in itertools.islice(naturals(), 5)])
print("first 3 over 100 :", list(itertools.islice((n for n in naturals() if n > 100), 3)))

---

### Where this leads: generators underpin `async`

A generator is a function that **suspends and resumes**, keeping its local state across
pauses. That is exactly what asynchronous code needs: pause here, let something else run,
come back when the data arrives.

Historically `asyncio` coroutines *were* generators — you wrote `@asyncio.coroutine` and
`yield from`. Python 3.5 gave them dedicated syntax (`async def` / `await`), and 3.11
removed the old decorator entirely, but the underlying machinery is the same suspend/resume
idea you have just learned.

```python
def countdown(n):          # generator: yields values
    while n > 0:
        yield n
        n -= 1

async def countdown(n):    # coroutine: awaits results
    while n > 0:
        await asyncio.sleep(1)
        n -= 1
```

Full treatment in **12 Multithreading** (`asyncio` / `async`-`await`).

---

## Common Mistakes & Pitfalls

1. **Consuming a generator twice.** It is exhausted after one pass and silently yields nothing the second time — no error, just empty results. Materialise with `list()` if you need more than one pass.
2. **Calling `len()` on a generator.** There is no length to know — it hasn't run yet.
3. **Expecting the body to run when you call the function.** Calling a generator function creates the generator and executes **nothing**. The first `next()` runs up to the first `yield`.
4. **Mixing `return` and `yield` and expecting the return value from iteration.** The value goes into `StopIteration.value`, not into the loop.
5. **Forgetting to `.send(None)` (or `next()`) before the first real `.send()`.** A fresh generator must be advanced to its first `yield` first.
6. **Using a generator where you need indexing or slicing.** `gen[0]` is a `TypeError`.
7. **Manually looping to re-yield a sub-generator** instead of using `yield from` — which also loses `send`/`throw` propagation.
8. **Building a huge list just to feed `sum()`/`any()`/`max()`.** Pass a generator expression.

## Best Practices

- Use a generator when the data is large, streamed, or infinite — anything you don't need all of at once.
- Use a generator **expression** for one-liners, a generator **function** when there's logic.
- Use `yield from` to delegate to another iterable — it is clearer and preserves the protocol.
- Chain generators into a pipeline; each stage stays lazy and memory stays flat.
- Return a generator from functions that read files or query APIs, so the caller controls how much is consumed.
- Reach for `itertools` before hand-rolling — `islice`, `chain`, `takewhile` cover most needs.
- Remember generators are one-shot: document it, or return a list if callers will re-iterate.

## Practice Exercises

Try these before moving on.

1. Write a generator yielding the first `n` triangular numbers.
2. Write an infinite generator of primes and use `itertools.islice` to take the first 20.
3. Build a three-stage pipeline over a log file: read lines -> parse -> filter errors. Confirm memory stays flat for a large file.
4. Write `flatten(nested)` using `yield from` recursively, handling arbitrary depth.
5. Write a generator that `return`s a summary value and show how to retrieve it from `StopIteration.value`.
6. Implement a running-average coroutine using `.send()`.
7. Compare `sys.getsizeof` and timing for a list comprehension vs a generator over 1,000,000 items.
8. Explain why `sum(gen)` works but `len(gen)` does not.